# Objetivo A Deteccion de Fraude en Apps Moviles con LightGBM

Este notebook implementa el Objetivo A para clasificacion binaria de fraude con metricas personalizadas de LightGBM enfocadas en reducir alertas falsas positivas en transacciones de apps moviles.

Columna objetivo: is_fraud

Metrica principal del negocio: false_positive_ratio igual a FP dividido entre TP mas FP.

## Configuracion del proyecto

Esta seccion importa librerias, fija la semilla aleatoria, crea la carpeta outputs y define constantes globales.

In [4]:
from pathlib import Path
import json
import pickle
import re
import unicodedata
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Global reproducibility settings
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Project paths
PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_CHECKS_DIR = OUTPUT_DIR / "01_data_checks"
EDA_DIR = OUTPUT_DIR / "02_eda"
FEATURES_DIR = OUTPUT_DIR / "03_features"
BASELINE_DIR = OUTPUT_DIR / "04_baseline"
CUSTOM_METRICS_DIR = OUTPUT_DIR / "05_custom_metrics"
TUNING_DIR = OUTPUT_DIR / "06_tuning"
FINAL_MODEL_DIR = OUTPUT_DIR / "07_final_model"
PLOTS_DIR = OUTPUT_DIR / "08_plots"
DELIVERY_DIR = OUTPUT_DIR / "09_delivery"
OUTPUT_INDEX_PATH = OUTPUT_DIR / "output_index.csv"

OUTPUT_SUBDIRS = {
    "data_checks": DATA_CHECKS_DIR,
    "eda": EDA_DIR,
    "features": FEATURES_DIR,
    "baseline": BASELINE_DIR,
    "custom_metrics": CUSTOM_METRICS_DIR,
    "tuning": TUNING_DIR,
    "final_model": FINAL_MODEL_DIR,
    "plots": PLOTS_DIR,
    "delivery": DELIVERY_DIR,
}


# Crea las carpetas de salida
def ensure_output_dirs():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for output_dir in OUTPUT_SUBDIRS.values():
        output_dir.mkdir(parents=True, exist_ok=True)


ensure_output_dirs()

# Main constants
# Define constantes principales
DATASET_FILE_NAME = "01_bo_vip_seed22_n100000.csv"
DATASET_PATTERN = "*01_bo_vip_seed22_n100000*.csv"
TARGET_COLUMN = "is_fraud"
ASSUMED_YEAR = 2025
TARGET_RECALL = 0.90
MOBILE_APP_COLUMN = "is_mobile_app_txn"

# Plot settings
# Configura graficas
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
warnings.filterwarnings("ignore")

print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")


Project directory: d:\Proyecto-DataSience-PlusTI
Output directory: d:\Proyecto-DataSience-PlusTI\outputs


## Descubrimiento de archivos

Esta seccion lista los archivos del proyecto, identifica el dataset principal y muestra vistas previas de archivos de metadatos cuando existen.

In [5]:
# Lista los archivos del proyecto
def get_file_inventory(project_dir):
    rows = []
    for path in sorted(project_dir.iterdir()):
        if path.is_file():
            rows.append(
                {
                    "file_name": path.name,
                    "suffix": path.suffix.lower(),
                    "size_bytes": path.stat().st_size,
                }
            )
    return pd.DataFrame(rows)


# Lee texto y lo convierte a ascii
def read_text_as_ascii(path, max_chars=5000):
    raw_bytes = path.read_bytes()
    for encoding_name in ["utf-8", "cp1252", "latin1"]:
        try:
            text = raw_bytes.decode(encoding_name)
            break
        except UnicodeDecodeError:
            text = ""
    if not text:
        text = raw_bytes.decode("latin1", errors="ignore")
    clean_text = text.encode("ascii", errors="ignore").decode("ascii")
    return clean_text[:max_chars]


# Muestra una vista previa de archivos tabulares
def preview_tabular_file(path, max_rows=5):
    try:
        if path.suffix.lower() == ".csv":
            sample = path.read_text(errors="ignore")[:4096]
            delimiter = ";" if sample.count(";") >= sample.count(",") else ","
            return pd.read_csv(path, sep=delimiter, nrows=max_rows).to_string()
        if path.suffix.lower() in [".xlsx", ".xls"]:
            return pd.read_excel(path, nrows=max_rows).to_string()
    except Exception as error:
        return f"Could not preview tabular file: {error}"
    return ""


# Muestra una vista previa de archivos pdf
def preview_pdf_file(path, max_chars=3000):
    try:
        import pypdf

        reader = pypdf.PdfReader(str(path))
        text_parts = []
        for page in reader.pages[:3]:
            text_parts.append(page.extract_text() or "")
        text = "\n".join(text_parts)
        return text.encode("ascii", errors="ignore").decode("ascii")[:max_chars]
    except Exception as error:
        return f"PDF preview unavailable: {error}"


# Muestra una vista previa de metadata
def preview_metadata_file(path):
    suffix = path.suffix.lower()
    if suffix in [".txt", ".md", ".json", ".yaml", ".yml"]:
        return read_text_as_ascii(path)
    if suffix in [".csv", ".xlsx", ".xls"]:
        return preview_tabular_file(path)
    if suffix == ".pdf":
        return preview_pdf_file(path)
    return ""


file_inventory = get_file_inventory(PROJECT_DIR)
display(file_inventory)

metadata_suffixes = {".txt", ".md", ".csv", ".xlsx", ".xls", ".json", ".pdf"}


# Detecta archivos de transacciones
def is_likely_transaction_dataset(file_name):
    lower_name = file_name.lower()
    return lower_name.endswith(".csv") and "seed" in lower_name and "n100000" in lower_name



,file_name,suffix,size_bytes
0,.gitignore,,4846
1,Copia de 01_bo_vip_seed22_n100000.csv,.csv,56111783
2,Copia de 02_br_privado_seed33_n100000.csv,.csv,56588931
3,Copia de 03_gt_estatal_seed3_n100000.csv,.csv,55910287
4,Descripciones de variables ISO 8583 Para Alumn...,.txt,11334
5,Indicaciones Trabajo Practico 1 UDV.pdf,.pdf,258487
6,LICENSE,,1101
7,ObjetivoA.ipynb,.ipynb,360354
8,README.md,.md,28
9,Regla.txt,.txt,185


## Carga de datos

Esta seccion carga el archivo CSV principal, normaliza los nombres de columnas a snake case y valida la columna objetivo.

In [6]:
# Convierte texto a ascii
def normalize_to_ascii(value):
    text = str(value)
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", errors="ignore").decode("ascii")
    return text


# Convierte texto a snake case
def to_snake_case(value):
    text = normalize_to_ascii(value)
    text = text.strip()
    text = re.sub(r"[^0-9a-zA-Z]+", "_", text)
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", text)
    text = re.sub(r"_+", "_", text)
    text = text.strip("_").lower()
    return text or "unnamed_column"


# Normaliza nombres de columnas
def normalize_column_names(columns):
    normalized_columns = []
    seen_counts = {}
    for column in columns:
        base_name = to_snake_case(column)
        count = seen_counts.get(base_name, 0)
        if count == 0:
            normalized_columns.append(base_name)
        else:
            normalized_columns.append(f"{base_name}_{count + 1}")
        seen_counts[base_name] = count + 1
    return normalized_columns


# Busca el dataset principal
def find_dataset_path(project_dir, dataset_file_name, dataset_pattern):
    exact_path = project_dir / dataset_file_name
    if exact_path.exists():
        return exact_path
    matches = sorted(project_dir.glob(dataset_pattern))
    if matches:
        warnings.warn(f"Exact dataset name not found. Using fallback file: {matches[0].name}")
        return matches[0]
    raise FileNotFoundError(f"Dataset file not found with pattern: {dataset_pattern}")


# Detecta el separador del csv
def detect_csv_separator(path):
    sample = path.read_text(errors="ignore")[:8192]
    counts = {separator: sample.count(separator) for separator in [";", ",", "\t", "|"]}
    best_separator = max(counts, key=counts.get)
    return best_separator if counts[best_separator] > 0 else ","


# Convierte el target a binario
def convert_target_to_int(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(int)
    normalized = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": 1,
        "false": 0,
        "1": 1,
        "0": 0,
        "yes": 1,
        "no": 0,
        "y": 1,
        "n": 0,
        "fraud": 1,
        "legit": 0,
        "legitimate": 0,
    }
    converted = normalized.map(mapping)
    if converted.isna().any():
        unknown_values = sorted(normalized[converted.isna()].dropna().unique().tolist())[:10]
        raise ValueError(f"Target has unsupported values: {unknown_values}")
    return converted.astype(int)


# Carga y valida el dataset
dataset_path = find_dataset_path(PROJECT_DIR, DATASET_FILE_NAME, DATASET_PATTERN)
csv_separator = detect_csv_separator(dataset_path)

print(f"Dataset path: {dataset_path}")
print(f"Detected separator: {repr(csv_separator)}")

raw_data = pd.read_csv(dataset_path, sep=csv_separator, low_memory=False)
original_columns = list(raw_data.columns)
raw_data.columns = normalize_column_names(raw_data.columns)
column_name_mapping = dict(zip(original_columns, raw_data.columns))

print(f"Loaded shape: {raw_data.shape}")
display(raw_data.head())

if TARGET_COLUMN not in raw_data.columns:
    raise ValueError(f"Required target column not found: {TARGET_COLUMN}")

raw_data[TARGET_COLUMN] = convert_target_to_int(raw_data[TARGET_COLUMN])
target_values = sorted(raw_data[TARGET_COLUMN].dropna().unique().tolist())
if target_values != [0, 1]:
    raise ValueError(f"Target must be binary 0 and 1. Found values: {target_values}")

with open(DATA_CHECKS_DIR / "column_name_mapping.json", "w", encoding="utf-8") as file:
    json.dump(column_name_mapping, file, indent=2, ensure_ascii=True)

print("Target validation passed")
print(raw_data[TARGET_COLUMN].value_counts(dropna=False).to_string())

Dataset path: d:\Proyecto-DataSience-PlusTI\Copia de 01_bo_vip_seed22_n100000.csv
Detected separator: ';'
Loaded shape: (100003, 66)


,transaction_id,bank_code,bank_name,bank_country,bank_tier,client_id,client_segment,channel,card_brand,pan_masked,...,amount_usd,is_international,distance_from_home_km,hour_local,day_of_week,approved,response_description,client_baseline_amount,client_home_city,is_fraud
0,7dd812b1-bd03-4d05-afc6-c318dcc9b651,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00001325,PLATINUM,POS,MASTERCARD,531270******3773,...,500.12,True,8717.0,20,Tue,True,Approved,2012.51,TARIJA,False
1,c08b49a6-889a-491a-a1f8-974526f7886d,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00000079,PRIVATE,ECOM,VISA,421250******5552,...,1898.93,False,4.9,20,Tue,True,Approved,1096.46,LAPAZ,False
2,b04f88bd-2e33-42e5-a3cf-d52ef22dd7d9,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00002344,INFINITE,ECOM,NaN,531270******6104,...,349.85,False,4.4,20,Tue,True,Approved,1528.37,SANTACRUZ,False
3,3a836c25-7a8c-473b-8141-3e84ba3f212d,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00002587,PLATINUM,ATM,VISA,479500******0288,...,345.58,True,3966.0,20,Tue,True,Approved,2483.34,SUCRE,False
4,be9956da-924f-4c68-aed8-f0c5d949e577,BO-VIP,BO-VIP,BO,vip,BO-VIP-CL-00000087,PRIVATE,POS,VISA,479500******0249,...,118.90,False,348.0,20,Tue,True,NaN,1334.55,SUCRE,False


Target validation passed
is_fraud
0    95084
1     4919


## EDA basico

Esta seccion revisa tipos de datos, valores faltantes, duplicados, cardinalidad, balance de la columna objetivo, estadisticas descriptivas y graficas basicas.